# B2-019-attention-transformers — Practice p14 — Solution

**Type:** proof · **Difficulty:** core · **Concepts:** causal-self-attention

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260808`  
**Qualified Book 1 prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C11-neural-training`  
**Remediation:** review the linked Book 1 units before continuing: [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb).

## Independent solution

Base case: the layer-zero representation at i is its input at i, hence depends only on positions through i. Induction step: assume every layer-\(\ell\) representation at source j depends only on inputs 0,...,j. Causal row i uses only j<=i, so every attended value depends only on inputs through i. Position-wise sublayers do not mix positions, and adding the residual at i introduces no later dependency; the invariant holds at layer \(\ell+1\). Post-softmax zeroing of j>i still removes future values, so causal independence remains, but it discards probability mass. For two scores (0,0), softmax is (1/2,1/2); zeroing the future entry gives (1/2,0), whose sum is 1/2. A reversed triangle admits j>i (for example row 0 reading column 1), so future values directly enter the output and causal independence fails.

In [ ]:
import numpy as np
SEED = 20260808
ATOL = 1e-12
RTOL = 1e-12
scores = np.array([[0.0, 0.0]], dtype=np.float64)
full_weights = np.exp(scores) / np.sum(np.exp(scores), axis=-1, keepdims=True)
postmasked = full_weights * np.array([[1.0, 0.0]], dtype=np.float64)
future_values = np.array([[0.0], [9.0]], dtype=np.float64)
causal_output = postmasked @ future_values
reversed_output = np.array([[0.0, 1.0]], dtype=np.float64) @ future_values

### Answer check

In [ ]:
np.testing.assert_allclose(full_weights, [[0.5, 0.5]], atol=ATOL, rtol=RTOL)
assert np.isclose(postmasked.sum(), 0.5, atol=ATOL, rtol=RTOL)
assert np.isclose(causal_output[0, 0], 0.0, atol=ATOL, rtol=RTOL)
assert np.isclose(reversed_output[0, 0], 9.0, atol=ATOL, rtol=RTOL)